In [ ]:
import os, sys, json, time, subprocess, re
from pathlib import Path

try:
    import pandas, numpy, sklearn, mlflow, fastapi, uvicorn, joblib
except Exception:
    import sys as _s, subprocess as _sp
    _sp.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'mlflow', 'fastapi', 'uvicorn', 'joblib', 'requests'])

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from fastapi import FastAPI
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rt = Path.cwd() / 'SinhVienColab'
rt.mkdir(exist_ok=True)
os.chdir(rt)
for d in ['data', 'models', 'logs']:
    Path(d).mkdir(exist_ok=True)
if not Path('churnDataset.csv').exists():
    try:
        from google.colab import files
        up = files.upload()
        for k in up:
            if k.endswith('.csv'):
                Path(k).rename('churnDataset.csv')
                break
    except Exception:
        pass

def cn(x):
    x = str(x).strip().lower()
    x = re.sub(r'[^a-z0-9]+', '_', x)
    return x.strip('_')

def cl(df):
    df = df.copy()
    df.columns = [cn(c) for c in df.columns]
    df = df.drop_duplicates()
    for c in df.select_dtypes(include='object').columns:
        df[c] = df[c].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    df['churn'] = pd.to_numeric(df['churn'], errors='coerce')
    return df

def fe(df):
    df = df.copy()
    df['spend_per_tenure'] = df['total_spend'] / df['tenure'].replace(0, np.nan)
    df['call_delay_ratio'] = df['support_calls'] / (df['payment_delay'] + 1)
    df['usage_recent_score'] = df['usage_frequency'] / (df['last_interaction'] + 1)
    df['spend_usage_ratio'] = df['total_spend'] / (df['usage_frequency'] + 1)
    return df.replace([np.inf, -np.inf], np.nan)

def miss_a(df):
    df = df.copy()
    ns = df.select_dtypes(include=np.number).columns
    cs = [c for c in df.columns if c not in ns]
    for c in ns:
        df[c] = df[c].fillna(df[c].median())
    for c in cs:
        m = df[c].mode(dropna=True)
        df[c] = df[c].fillna(m.iloc[0] if len(m) else 'unknown')
    return df

def miss_b(df):
    df = df.copy()
    ns = list(df.select_dtypes(include=np.number).columns)
    cs = [c for c in df.columns if c not in ns]
    df[ns] = KNNImputer(n_neighbors=5).fit_transform(df[ns])
    if cs:
        df[cs] = SimpleImputer(strategy='most_frequent').fit_transform(df[cs])
    return df

def out(df):
    df = df.copy()
    ns = [c for c in df.select_dtypes(include=np.number).columns if c not in ['churn', 'customerid', 'customer_id']]
    mk = pd.Series(True, index=df.index)
    for c in ns:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iq = q3 - q1
        if iq > 0:
            mk &= df[c].between(q1 - 1.5 * iq, q3 + 1.5 * iq)
    rs = df.loc[mk].reset_index(drop=True)
    return rs if len(rs) else df.reset_index(drop=True)

def oh():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

df = pd.read_csv('churnDataset.csv')
df = fe(cl(df))
a = out(miss_a(df))
b = out(miss_b(df))
a.to_csv('data/processed_median_mode.csv', index=False)
b.to_csv('data/processed_knn_mode.csv', index=False)
tb = b.copy()
tb['churn'] = tb['churn'].round().astype(int)
fs = [c for c in tb.columns if c not in ['churn', 'customerid', 'customer_id']]
ns = [c for c in fs if pd.api.types.is_numeric_dtype(tb[c])]
cs = [c for c in fs if c not in ns]
tb.to_csv('data/processed_churn.csv', index=False)
Path('data/columns.json').write_text(json.dumps({'features': fs, 'num': ns, 'cat': cs}, indent=2), encoding='utf-8')
st = {c: {'mean': float(tb[c].mean()), 'std': float(tb[c].std() or 1.0)} for c in ns}
Path('data/train_stats.json').write_text(json.dumps(st, indent=2), encoding='utf-8')
print('data ok', tb.shape[0], tb.shape[1])

mlflow.set_tracking_uri(Path('mlruns').resolve().as_uri())
mlflow.set_experiment('churn')
x = tb[fs]
y = tb['churn'].astype(int)
xtr, xt, ytr, yt = train_test_split(x, y, test_size=0.3, stratify=y, random_state=42)
xv, xte, yv, yte = train_test_split(xt, yt, test_size=0.5, stratify=yt, random_state=42)
ms = {'lr': LogisticRegression(max_iter=1000, class_weight='balanced'), 'rf': RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced', n_jobs=-1)}
rs = []
for nm, al in ms.items():
    with mlflow.start_run(run_name=nm) as rn:
        m = Pipeline([('pp', ColumnTransformer([('n', StandardScaler(), ns), ('c', oh(), cs)])), ('md', al)])
        m.fit(xtr, ytr)
        pv = m.predict(xv)
        pt = m.predict(xte)
        va = {'accuracy': float(accuracy_score(yv, pv)), 'f1': float(f1_score(yv, pv))}
        te = {'accuracy': float(accuracy_score(yte, pt)), 'f1': float(f1_score(yte, pt))}
        pm = al.get_params()
        mlflow.log_params({k: v for k, v in pm.items() if isinstance(v, (str, int, float, bool, type(None)))})
        mlflow.log_metric('val_accuracy', va['accuracy'])
        mlflow.log_metric('val_f1', va['f1'])
        mlflow.log_metric('test_accuracy', te['accuracy'])
        mlflow.log_metric('test_f1', te['f1'])
        mlflow.sklearn.log_model(m, 'model')
        rs.append({'name': nm, 'run_id': rn.info.run_id, 'val_f1': va['f1'], 'test_f1': te['f1']})
        print(nm, round(va['accuracy'], 4), round(va['f1'], 4), round(te['accuracy'], 4), round(te['f1'], 4))
best = max(rs, key=lambda r: r['val_f1'])
bm = mlflow.sklearn.load_model(f"runs:/{best['run_id']}/model")
joblib.dump(bm, 'models/best_model.pkl')
Path('models/runs.json').write_text(json.dumps(rs, indent=2), encoding='utf-8')
Path('models/best_run.json').write_text(json.dumps(best, indent=2), encoding='utf-8')
Path('models/sample_input.json').write_text(json.dumps(x.iloc[0].to_dict(), indent=2), encoding='utf-8')
print('best', best['name'], round(best['val_f1'], 4))

from mlflow.tracking import MlflowClient
cnm = 'churn_model'
clt = MlflowClient()
vs = []
for r in sorted(rs, key=lambda z: z['val_f1'], reverse=True):
    mv = mlflow.register_model(f"runs:/{r['run_id']}/model", cnm)
    vs.append({'name': r['name'], 'version': mv.version, 'val_f1': r['val_f1']})
for i, v in enumerate(vs):
    sg = 'Production' if i == 0 else 'Staging'
    try:
        clt.transition_model_version_stage(cnm, v['version'], sg, archive_existing_versions=False)
    except Exception:
        clt.set_registered_model_alias(cnm, sg.lower(), v['version'])
    v['stage'] = sg
Path('models/registry.json').write_text(json.dumps(vs, indent=2), encoding='utf-8')
print('production', vs[0]['name'], vs[0]['version'])

api = """
import json, re
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from fastapi import FastAPI
app = FastAPI(title='churn api')
def cn(x):
    x = str(x).strip().lower()
    x = re.sub(r'[^a-z0-9]+', '_', x)
    return x.strip('_')
def fe(df):
    df = df.copy()
    df['spend_per_tenure'] = df['total_spend'] / df['tenure'].replace(0, np.nan)
    df['call_delay_ratio'] = df['support_calls'] / (df['payment_delay'] + 1)
    df['usage_recent_score'] = df['usage_frequency'] / (df['last_interaction'] + 1)
    df['spend_usage_ratio'] = df['total_spend'] / (df['usage_frequency'] + 1)
    return df.replace([np.inf, -np.inf], np.nan)
def prep(x):
    df = pd.DataFrame([x])
    df.columns = [cn(c) for c in df.columns]
    df = fe(df)
    cm = json.loads(Path('data/columns.json').read_text(encoding='utf-8'))
    for c in cm['features']:
        if c not in df.columns:
            df[c] = 0
    return df[cm['features']]
def drf(df):
    st = json.loads(Path('data/train_stats.json').read_text(encoding='utf-8'))
    rs = {}
    for c, v in st.items():
        if c in df.columns:
            sd = v['std'] if v['std'] else 1.0
            z = abs(float(df[c].mean()) - v['mean']) / sd
            if z > 3.0:
                rs[c] = round(z, 4)
    return rs
@app.post('/predict')
def predict(x: dict):
    m = joblib.load('models/best_model.pkl')
    tb = prep(x)
    y = int(m.predict(tb)[0])
    p = float(m.predict_proba(tb)[0][1]) if hasattr(m, 'predict_proba') else float(y)
    d = drf(tb)
    with open('logs/predictions.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps({'time': datetime.utcnow().isoformat(), 'input': x, 'label': y, 'probability': p, 'drift': d}, ensure_ascii=False) + '\n')
    return {'label': y, 'probability': p, 'drift': bool(d), 'drift_features': d}
"""
Path('app.py').write_text(api, encoding='utf-8')
subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', 'mlruns', '--host', '0.0.0.0', '--port', '5000'])
subprocess.Popen(['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(4)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(5000)
    output.serve_kernel_port_as_window(8000)
except Exception:
    print('MLflow UI: http://127.0.0.1:5000')
    print('API: http://127.0.0.1:8000')
print('curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d @models/sample_input.json')
